# Needle 2 — LoRA fine-tune on Colab

Upload a `data.jsonl`, get back an adapter and a `.cact` the app can load.

Everything here runs on Colab's GPU because the Mac does not have one worth
using: JAX's Metal backend is experimental, and on an M4 Max it managed an
out-of-memory fault, a poisoned command queue, and a run that sat in XLA
compilation until it was killed. A T4 is slower on paper and finishes.

**Runtime → Change runtime type → T4 GPU** before you start. The next cell
refuses to continue without one.

The input is the trainer's format, one JSON object per line:

```json
{"query": "...", "reasoning": "...",
 "answers": [{"name": "log_food", "arguments": {"food": "greek yoghurt"}}],
 "tools": [ ...the schemas the model was shown... ]}
```

`scripts/needle2-finetune/generate.py` writes exactly this.


## 1. Check the hardware

Two separate questions — whether a GPU is attached, and whether JAX can see
it. A runtime with a T4 and a CPU-only jaxlib trains at about a fortieth of
the speed and says nothing about it.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU'

import subprocess, sys
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit(
        'No GPU attached. Runtime → Change runtime type → T4 GPU, then rerun.'
    )


## 2. Install

`cactus-needle[gpu]` pulls `jax[cuda12]`. Takes two or three minutes, and
Colab will want a session restart afterwards on some images — if the JAX
check below reports `cpu`, restart the runtime and run this cell again.


In [ ]:
!pip install -q 'cactus-needle[gpu]'

import jax
print('jax', jax.__version__, '·', jax.devices())
if jax.devices()[0].platform != 'gpu':
    print('\nJAX is on the CPU. Runtime → Restart session, then rerun this cell.')


## 3. Upload your data

Pick the `.jsonl`. Several files are fine — they are concatenated in the
order you choose them.


In [ ]:
from google.colab import files
import json, pathlib

uploaded = files.upload()
rows = []
for name, blob in uploaded.items():
    for line in blob.decode().splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))

pathlib.Path('raw.jsonl').write_text(
    '\n'.join(json.dumps(row) for row in rows) + '\n'
)
print(f'{len(rows)} examples from {len(uploaded)} file(s)')
print('multi-step:', sum(1 for r in rows if len(r.get("answers", [])) >= 2))
print('empty-call:', sum(1 for r in rows if r.get("answers") == []))


## 4. Trim the tool lists — do not skip this

This is the cell that decides whether the run means anything.

The trainer masks everything except the answer and then truncates at
`--max-len`. Our first attempt inlined all fifteen tool schemas into every
example, which rendered as a ~5,000-token prompt against a 1,024 cap — so
the truncation landed four thousand tokens *before* the answer began, the
loss mask was all zeros, and sixteen steps reported a loss of exactly
`0.0000`. It trained on nothing and looked like it was working.

Five tools is also what the app declares at inference (`NEEDLE_FAMILIES`),
so this makes training match the thing being trained for.


In [ ]:
KEEP = 5

trimmed = []
for row in rows:
    called = {a['name'] for a in row.get('answers', [])}
    tools = row.get('tools', [])
    # The tools the answer names first, then whatever fills the slate —
    # a lineup with no distractors teaches the model there are none.
    wanted = [t for t in tools if t['name'] in called]
    wanted += [t for t in tools if t['name'] not in called][: KEEP - len(wanted)]
    trimmed.append({**row, 'tools': wanted[:KEEP]})

with open('prepared.jsonl', 'w') as handle:
    for row in trimmed:
        handle.write(json.dumps(row) + '\n')
print(f'{len(trimmed)} rows → prepared.jsonl')


## 5. Measure the sequence length

Then pass it in explicitly. `fit_max_len` rounds up to a power of two and
clamps to whatever you give it, so handing it the true longest example
avoids padding every row out to 2048 and doubling the activation memory for
nothing.

If `answers survive` is anything but 100%, stop and trim harder — those rows
will train on zero tokens.


In [ ]:
from needle.model.finetune import render_example, get_tokenizer

tok = get_tokenizer(8192)
lengths, survivors = [], 0
for row in trimmed:
    prompt, target = render_example(row)
    p, t = len(tok.encode(prompt)), len(tok.encode(target))
    lengths.append(p + t + 2)
    if p + t + 2 <= 2048:
        survivors += 1

lengths.sort()
MAX_LEN = min(2048, lengths[-1] + 8)
print(f'median {lengths[len(lengths)//2]}  p90 {lengths[int(len(lengths)*0.9)]}  max {lengths[-1]}')
print(f'answers survive: {survivors}/{len(lengths)}  ({100*survivors//len(lengths)}%)')
print(f'--max-len {MAX_LEN}')


## 6. Train

Two epochs is a starting point, not a recommendation. The epoch count sets
the cosine schedule, so three epochs is a different run from two epochs left
going — you cannot hedge by stopping early. Watch the validation loss the
trainer prints: still falling at the end means run longer, turned upward
means run shorter.

**Watch the first two step lines.** A real number means it is learning.
`0.0000` means the mask is empty and cell 4 did not do its job.


In [ ]:
EPOCHS = 2
BATCH_SIZE = 8
LORA_RANK = 16

!needle finetune prepared.jsonl \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lora-rank {LORA_RANK} \
  --max-len {MAX_LEN} \
  --out onerep-lora.pkl

# The base checkpoint is fetched from Hugging Face on first use; there is
# nothing to upload. The adapter is pickled once, after the final step, so
# the file appearing at all is the signal that the run finished.


## 7. Merge and export

`needle build` merges the adapter into the base and writes the W4A8 blob the
C engine memory-maps. Same 13.7 MB as the stock weights — a LoRA adds no
size once merged.


In [ ]:
!needle build checkpoints/needle2.pkl --lora onerep-lora.pkl --out needle2-onerep.cact
!ls -lh needle2-onerep.cact onerep-lora.pkl


## 8. Download

Then, on the machine with the repo:

```sh
cp ~/Downloads/needle2-onerep.cact scripts/needle2-finetune/
bun run needle:tuned      # copies it to apps/mobile/public/needle/
```

`apps/mobile/src/lib/needle.ts` already asks for `needle2-onerep.cact` by
name, so replacing the file is the whole deployment.

Before you ship it, measure it against the weights you are replacing —
portions, meal inference, and above all negation. The last adapter read
"half a chicken breast" correctly and also logged greek yoghurt for "I
skipped lunch today".


In [ ]:
files.download('needle2-onerep.cact')
files.download('onerep-lora.pkl')
